# 🏆 Benchmark — Euro 2024 GK Build-up Analysis
### All Teams Compared · Spain Highlighted

---

**Objective:** Benchmark every nation's goal-kick build-up across all Euro 2024 matches, using the same KPI framework developed in the Spain opposition analysis notebook. Spain is highlighted in every chart so the coaching staff can instantly see where Spain sits relative to the tournament field.

**KPIs benchmarked (Top 5 bar charts, PPDA-dashboard style):**

| # | KPI | What it measures |
|---|-----|------------------|
| 1 | **Short Distribution %** | Preference for short goal kicks (zones 1–6) |
| 2 | **Long Distribution %** | Preference for long goal kicks (zones 7–18) |
| 3 | **Positive Outcome %** | How often the team retains possession after a goal kick |
| 4 | **Progression Rate (P2 + P3 %)** | How often the team reaches the final third or creates a shot |
| 5 | **Danger Conceded (N2 + N3 %)** | How often the opponent enters the box or shoots after recovering |
| 6 | **6-Tier Outcome Breakdown** | Full P1–P3 / N1–N3 stacked bar for all teams |

**Chart style:** Horizontal bar chart replicating the PPDA ranking from the Serie A dashboard.  
**Highlight colour:** Spain bars in `#8a1f33` (deep red), all others in `#4a6274` (neutral blue-grey).

In [1]:
# ══════════════════════════════════════════════════════════════
# CELL 1 — Imports & Configuration
# ══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ast
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──
DATA_DIR = '../AnalisisDeJuego_Europea_Marzo26'
ALL_EVENTS = f'{DATA_DIR}/SB_Euro24_allevents.xlsx'
MATCH_DETAILS = f'{DATA_DIR}/SB_Euro24_matchdetails.xlsx'

# ── Highlight team ──
HIGHLIGHT_TEAM = 'Spain'

# ── Analysis Configuration ──
POSSESSION_WINDOW_SEC = 15.0

# ── StatsBomb pitch dimensions ──
SB_X_MAX = 120.0
SB_Y_MAX = 80.0
ROW_WIDTH = SB_X_MAX / 6    # 20.0
COL_WIDTH = SB_Y_MAX / 3    # 26.67

SHORT_ZONES = set(range(1, 7))
LONG_ZONES  = set(range(7, 19))
FINAL_THIRD_X = SB_X_MAX / 6 * 4  # 80.0
BOX_X = SB_X_MAX / 6 * 5          # 100.0

# ── PPDA-dashboard bar chart palette ──
NEUTRAL_COLOR   = '#4a6274'
HIGHLIGHT_COLOR = '#8a1f33'

# ── Non-play / shot / contested constants ──
NON_PLAY_EVENTS = frozenset({
    'starting xi', 'half start', 'half end', 'substitution',
    'player on', 'player off', 'tactical shift', 'referee ball-drop',
    'injury stoppage', 'bad behaviour', 'shield',
})

print('✅ Configuration loaded')
print(f'   Highlight team: {HIGHLIGHT_TEAM}')
print(f'   Bar chart style: PPDA-dashboard horizontal bars')

✅ Configuration loaded
   Highlight team: Spain
   Bar chart style: PPDA-dashboard horizontal bars


In [2]:
# ══════════════════════════════════════════════════════════════
# CELL 2 — Load Data & Parse Locations
# ══════════════════════════════════════════════════════════════

df_all = pd.read_excel(ALL_EVENTS)
df_matches = pd.read_excel(MATCH_DETAILS)

def parse_location(val):
    if pd.isna(val):
        return None, None
    try:
        if isinstance(val, str):
            coords = ast.literal_eval(val)
        else:
            coords = val
        return float(coords[0]), float(coords[1])
    except:
        return None, None

df_all[['loc_x', 'loc_y']] = df_all['location'].apply(
    lambda v: pd.Series(parse_location(v))
)
df_all[['pass_end_x', 'pass_end_y']] = df_all['pass_end_location'].apply(
    lambda v: pd.Series(parse_location(v))
)
df_all = df_all.sort_values(
    ['match_id', 'period', 'minute', 'second', 'index']
).reset_index(drop=True)

def parse_timestamp_to_seconds(ts, minute, period):
    if pd.notna(ts):
        try:
            parts = str(ts).split(':')
            if len(parts) == 3:
                return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
            elif len(parts) == 2:
                return int(parts[0]) * 60 + float(parts[1])
        except:
            pass
    return float(minute) * 60 if pd.notna(minute) else 0.0

df_all['time_seconds'] = df_all.apply(
    lambda r: parse_timestamp_to_seconds(r['timestamp'], r['minute'], r['period']), axis=1
)

# ── Build team-match lookup ──
all_teams = sorted(
    set(df_matches['home_team'].tolist() + df_matches['away_team'].tolist())
)

print(f'✅ Loaded {len(df_all):,} events across {df_all["match_id"].nunique()} matches')
print(f'   {len(all_teams)} teams in the tournament')
print(f'   Teams: {", ".join(all_teams)}')

KeyboardInterrupt: 

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 3 — Core Analysis Functions (identical to Spain notebook)
# ══════════════════════════════════════════════════════════════

def xy_to_zone(x: float, y: float) -> int:
    x = max(0.0, min(SB_X_MAX, float(x)))
    y = max(0.0, min(SB_Y_MAX, float(y)))
    row = min(int(x / ROW_WIDTH), 5)
    col = min(int(y / COL_WIDTH), 2)
    return row * 3 + col + 1


def _is_play_event(event_type: str) -> bool:
    return event_type.lower() not in NON_PLAY_EVENTS and event_type != ''


def _elapsed_seconds(ref_row, row):
    if ref_row['period'] != row['period']:
        return POSSESSION_WINDOW_SEC + 1
    return row['time_seconds'] - ref_row['time_seconds']


def _find_first_receiver(gk_idx, match_df, team):
    for j in range(gk_idx + 1, min(gk_idx + 25, len(match_df))):
        row = match_df.iloc[j]
        et = str(row.get('type', '')).strip()
        if not _is_play_event(et):
            continue
        if et in ('Carry', 'Pressure'):
            continue
        if row['team'] == team:
            rx, ry = row['loc_x'], row['loc_y']
            if pd.notna(rx) and pd.notna(ry):
                zone = xy_to_zone(rx, ry)
            else:
                zone = 0
            return {
                'iloc': j,
                'player': str(row.get('player', '?')),
                'x': rx, 'y': ry,
                'zone': zone,
                'event_type': et,
            }
        else:
            return None
    return None


def _track_opponent_danger(start_idx, match_df, team):
    entered_box = False
    for j in range(start_idx, min(start_idx + 40, len(match_df))):
        row = match_df.iloc[j]
        et = str(row.get('type', '')).strip()
        if not _is_play_event(et.lower()):
            continue
        if row.get('possession_team') == team and row['team'] == team:
            break
        if row['team'] != team:
            if et == 'Shot':
                return 'N3'
            ox = row.get('loc_x')
            if pd.notna(ox) and float(ox) >= BOX_X:
                entered_box = True
    return 'N2' if entered_box else 'N1'


def _extract_chain_and_outcome(gk_idx, recv_idx, match_df, team):
    """Extract chain and classify outcome — parameterised by team."""
    gk_row = match_df.iloc[gk_idx]

    if recv_idx is None:
        pass_outcome = str(gk_row.get('pass_outcome', '')).strip()
        if pass_outcome in ('Incomplete', 'Out'):
            return 'negative', 'N1'
        for j in range(gk_idx + 1, min(gk_idx + 30, len(match_df))):
            row = match_df.iloc[j]
            et = str(row.get('type', '')).strip()
            if not _is_play_event(et):
                continue
            if et in ('Pressure', 'Carry'):
                continue
            if row['team'] != team:
                return 'negative', _track_opponent_danger(j, match_df, team)
            break
        return 'negative', 'N1'

    ref_row = match_df.iloc[recv_idx]
    reached_final_third = False
    rx = ref_row.get('loc_x')
    if pd.notna(rx) and float(rx) >= FINAL_THIRD_X:
        reached_final_third = True

    OPPONENT_CONSTRUCTIVE = frozenset({'Pass', 'Ball Receipt*', 'Carry', 'Dribble', 'Shot'})
    opp_consecutive = 0

    j = recv_idx + 1
    while j < len(match_df):
        row = match_df.iloc[j]
        elapsed = _elapsed_seconds(ref_row, row)
        et = str(row.get('type', '')).strip()
        if not _is_play_event(et.lower()):
            j += 1
            continue

        is_team = row['team'] == team
        poss_team = row.get('possession_team')

        if is_team:
            opp_consecutive = 0
            if et == 'Shot':
                return 'positive', 'P3'
            ex = row.get('loc_x')
            if pd.notna(ex) and float(ex) >= FINAL_THIRD_X:
                reached_final_third = True
            if elapsed >= POSSESSION_WINDOW_SEC:
                return 'positive', 'P2' if reached_final_third else 'P1'
            j += 1
            continue

        if et == 'Pressure':
            j += 1
            continue

        if et in OPPONENT_CONSTRUCTIVE:
            opp_consecutive += 1

        if et == 'Foul Committed':
            if opp_consecutive == 0:
                return 'positive', 'P2' if reached_final_third else 'P1'
            else:
                return 'negative', _track_opponent_danger(j, match_df, team)

        if poss_team != team:
            return 'negative', _track_opponent_danger(j, match_df, team)

        if opp_consecutive >= 2:
            return 'negative', _track_opponent_danger(j, match_df, team)

        j += 1

    return 'positive', 'P2' if reached_final_third else 'P1'


print('✅ Analysis functions defined (parameterised for any team)')

✅ Analysis functions defined (parameterised for any team)


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4 — Run Analysis for ALL Teams
# ══════════════════════════════════════════════════════════════

all_team_distributions = []

for team in all_teams:
    # Get matches where this team played
    team_matches = df_matches[
        (df_matches['home_team'] == team) | (df_matches['away_team'] == team)
    ]
    team_match_ids = team_matches['match_id'].tolist()

    for match_id in team_match_ids:
        match_df = df_all[df_all['match_id'] == match_id].copy()
        match_df = match_df.sort_values(
            ['period', 'minute', 'second', 'index']
        ).reset_index(drop=True)

        match_info = team_matches[team_matches['match_id'] == match_id].iloc[0]
        opponent = (
            match_info['away_team']
            if match_info['home_team'] == team
            else match_info['home_team']
        )
        stage = match_info['competition_stage']

        for idx in range(len(match_df)):
            row = match_df.iloc[idx]
            if (str(row.get('type', '')).strip() != 'Pass' or
                str(row.get('pass_type', '')).strip() != 'Goal Kick' or
                row.get('team') != team):
                continue

            receiver = _find_first_receiver(idx, match_df, team)

            if receiver is not None and receiver['zone'] > 0:
                recv_zone = receiver['zone']
                recv_idx = receiver['iloc']
            else:
                end_x, end_y = row.get('pass_end_x'), row.get('pass_end_y')
                if pd.notna(end_x) and pd.notna(end_y):
                    recv_zone = xy_to_zone(float(end_x), float(end_y))
                else:
                    recv_zone = 0
                recv_idx = None

            outcome, granular = _extract_chain_and_outcome(
                idx, recv_idx, match_df, team
            )

            if recv_zone in SHORT_ZONES:
                pass_type = 'short'
            elif recv_zone in LONG_ZONES:
                pass_type = 'long'
            else:
                length = row.get('pass_length')
                pass_type = (
                    'long' if pd.notna(length) and float(length) > 32.0 else 'short'
                )

            all_team_distributions.append({
                'team': team,
                'match_id': match_id,
                'opponent': opponent,
                'stage': stage,
                'pass_type': pass_type,
                'recv_zone': recv_zone,
                'outcome': outcome,
                'granular_outcome': granular,
            })

df_dist = pd.DataFrame(all_team_distributions)
print(f'✅ Analysed {len(df_dist):,} goal kicks across {len(all_teams)} teams')
print(f'   Goal kicks per team (top 10):')
print(df_dist.groupby('team').size().sort_values(ascending=False).head(10).to_string())

✅ Analysed 799 goal kicks across 24 teams
   Goal kicks per team (top 10):
team
England        52
Spain          51
Slovenia       47
Romania        47
Georgia        45
Belgium        42
Turkey         41
Netherlands    40
Poland         38
France         35


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 5 — Build KPI Summary Table (one row per team)
# ══════════════════════════════════════════════════════════════

team_kpis = []

for team in all_teams:
    tdf = df_dist[df_dist['team'] == team]
    total = len(tdf)
    if total == 0:
        continue

    n_matches = tdf['match_id'].nunique()
    short_n = (tdf['pass_type'] == 'short').sum()
    long_n  = (tdf['pass_type'] == 'long').sum()

    pos_n = (tdf['outcome'] == 'positive').sum()
    neg_n = (tdf['outcome'] == 'negative').sum()

    p1 = (tdf['granular_outcome'] == 'P1').sum()
    p2 = (tdf['granular_outcome'] == 'P2').sum()
    p3 = (tdf['granular_outcome'] == 'P3').sum()
    n1 = (tdf['granular_outcome'] == 'N1').sum()
    n2 = (tdf['granular_outcome'] == 'N2').sum()
    n3 = (tdf['granular_outcome'] == 'N3').sum()

    team_kpis.append({
        'team': team,
        'matches': n_matches,
        'total_gk': total,
        'gk_per_match': round(total / n_matches, 1),
        'short_n': short_n,
        'long_n': long_n,
        'short_pct': round(short_n / total * 100, 1),
        'long_pct': round(long_n / total * 100, 1),
        'positive_n': pos_n,
        'negative_n': neg_n,
        'positive_pct': round(pos_n / total * 100, 1),
        'negative_pct': round(neg_n / total * 100, 1),
        'P1': p1, 'P2': p2, 'P3': p3,
        'N1': n1, 'N2': n2, 'N3': n3,
        'progression_pct': round((p2 + p3) / total * 100, 1),
        'danger_conceded_pct': round((n2 + n3) / total * 100, 1),
    })

kpi_df = pd.DataFrame(team_kpis).sort_values('team').reset_index(drop=True)

# ── Tournament stage reached (Euro 2024 results) ──
# 3 GP = Group stage exit, 4 GP = Round of 16, 5 GP = Quarter-final,
# 6 GP = Semi-final, 7 GP = Final
STAGE_MAP = {
    3: 'Group Stage',
    4: 'Round of 16',
    5: 'Quarter-Final',
    6: 'Semi-Final',
    7: 'Final',
}
kpi_df['stage_reached'] = kpi_df['matches'].map(STAGE_MAP).fillna('Group Stage')

# Filter to teams with ≥ 3 goal kicks for meaningful comparison
MIN_GK = 3
kpi_df = kpi_df[kpi_df['total_gk'] >= MIN_GK].reset_index(drop=True)

print(f'✅ KPI summary for {len(kpi_df)} teams (min {MIN_GK} goal kicks)')
print(f'\n⚠️  IMPORTANT — Tournament context:')
print(f'   Euro 2024 is a short knockout tournament (3–7 matches per team).')
print(f'   Teams that went further have larger sample sizes → more reliable KPIs.')
print(f'   GP (Games Played) is shown next to every team name in the charts.\n')

print(f'📋 Matches played per team:')
for _, r in kpi_df.sort_values('matches', ascending=False).iterrows():
    print(f"   {r['team']:18s}  {int(r['matches'])} GP  ({r['stage_reached']})  — {int(r['total_gk'])} goal kicks")

print(f'\n📋 Full KPI table:')
display_cols = [
    'team', 'matches', 'stage_reached', 'total_gk', 'gk_per_match',
    'short_pct', 'positive_pct', 'progression_pct', 'danger_conceded_pct',
]
print(kpi_df[display_cols].to_string(index=False))

✅ KPI summary for 24 teams (min 3 goal kicks)

⚠️  IMPORTANT — Tournament context:
   Euro 2024 is a short knockout tournament (3–7 matches per team).
   Teams that went further have larger sample sizes → more reliable KPIs.
   GP (Games Played) is shown next to every team name in the charts.

📋 Matches played per team:
   England             7 GP  (Final)  — 52 goal kicks
   Spain               7 GP  (Final)  — 51 goal kicks
   Netherlands         6 GP  (Semi-Final)  — 40 goal kicks
   France              6 GP  (Semi-Final)  — 35 goal kicks
   Turkey              5 GP  (Quarter-Final)  — 41 goal kicks
   Switzerland         5 GP  (Quarter-Final)  — 33 goal kicks
   Portugal            5 GP  (Quarter-Final)  — 35 goal kicks
   Germany             5 GP  (Quarter-Final)  — 21 goal kicks
   Austria             4 GP  (Round of 16)  — 25 goal kicks
   Romania             4 GP  (Round of 16)  — 47 goal kicks
   Slovenia            4 GP  (Round of 16)  — 47 goal kicks
   Georgia             4

---
## 0. Tournament Context — Games Played per Team

Euro 2024 is a **direct-elimination tournament** after the group stage. Only 2 teams reach the final (7 GP). Most teams play just 3–4 matches. This means:

- **Sample sizes vary enormously** — Spain (7 GP, 51 GKs) vs Scotland (3 GP, 27 GKs)
- KPIs from 3-match teams should be read with caution
- The chart below shows how far each team went, colour-coded by **tournament stage**

| GP | Stage |
|----|-------|
| 3 | Group Stage exit |
| 4 | Round of 16 |
| 5 | Quarter-Final |
| 6 | Semi-Final |
| 7 | Final |

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 6b — Tournament Context: Games Played & Goal Kicks
# ══════════════════════════════════════════════════════════════

# ── Stage colour palette ──
STAGE_COLORS = {
    'Group Stage':    '#6b7280',   # grey
    'Round of 16':    '#3b82f6',   # blue
    'Quarter-Final':  '#f59e0b',   # amber
    'Semi-Final':     '#8b5cf6',   # purple
    'Final':          '#ef4444',   # red
}

# Sort by matches desc, then alphabetically
ctx = kpi_df.sort_values(['matches', 'team'], ascending=[True, True]).copy()

# Build colour array
bar_colors = [
    STAGE_COLORS.get(s, '#6b7280') for s in ctx['stage_reached']
]

fig_ctx = go.Figure()

fig_ctx.add_trace(go.Bar(
    x=ctx['matches'],
    y=ctx['team'],
    orientation='h',
    marker=dict(color=bar_colors, line=dict(width=0)),
    text=[
        f"  {int(m)} GP · {int(gk)} GKs · {s}"
        for m, gk, s in zip(ctx['matches'], ctx['total_gk'], ctx['stage_reached'])
    ],
    textposition='outside',
    textfont=dict(size=10, color='#c8d0d8'),
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Games Played: %{x}<br>'
        '<extra></extra>'
    ),
))

# Highlight Spain bar border
spain_idx = ctx[ctx['team'] == HIGHLIGHT_TEAM].index
if len(spain_idx) > 0:
    pos_in_list = list(ctx['team']).index(HIGHLIGHT_TEAM)
    fig_ctx.add_annotation(
        x=ctx.loc[spain_idx[0], 'matches'] + 0.15,
        y=HIGHLIGHT_TEAM,
        text='🇪🇸',
        showarrow=False,
        font=dict(size=14),
        xanchor='right',
    )

fig_ctx.update_layout(
    template='plotly_dark',
    title=dict(
        text='📋 Tournament Context — Games Played per Team'
             '<br><sup style="color:rgba(255,255,255,0.55)">'
             'Euro 2024 · Colour = stage reached · '
             'More games → larger sample → more reliable KPIs</sup>',
        font=dict(size=15, color='white'),
    ),
    xaxis=dict(
        title='Games Played',
        gridcolor='rgba(255,255,255,0.06)',
        zeroline=False,
        dtick=1,
        range=[0, 9],
    ),
    yaxis=dict(title='', tickfont=dict(size=10)),
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    height=max(500, len(ctx) * 24),
    margin=dict(l=130, r=200, t=70, b=40),
    showlegend=False,
    bargap=0.25,
)

# Legend as annotations for stage colours
legend_items = list(STAGE_COLORS.items())
for i, (stage, color) in enumerate(legend_items):
    fig_ctx.add_annotation(
        x=1.0, y=1.0 - i * 0.07,
        xref='paper', yref='paper',
        text=f'<span style="color:{color}">■</span> {stage}',
        showarrow=False,
        font=dict(size=10, color='#c8d0d8'),
        xanchor='left',
    )

fig_ctx.show()

# ── Summary stats ──
print(f'\n📊 Tournament distribution:')
for stage in ['Final', 'Semi-Final', 'Quarter-Final', 'Round of 16', 'Group Stage']:
    n = (kpi_df['stage_reached'] == stage).sum()
    if n > 0:
        teams = ', '.join(kpi_df[kpi_df['stage_reached'] == stage]['team'].tolist())
        gp = STAGE_MAP[[k for k, v in STAGE_MAP.items() if v == stage][0]]
        print(f'   {stage:16s}: {n} teams — {teams}')


📊 Tournament distribution:
   Final           : 2 teams — England, Spain
   Semi-Final      : 2 teams — France, Netherlands
   Quarter-Final   : 4 teams — Germany, Portugal, Switzerland, Turkey
   Round of 16     : 8 teams — Austria, Belgium, Denmark, Georgia, Italy, Romania, Slovakia, Slovenia
   Group Stage     : 8 teams — Albania, Croatia, Czech Republic, Hungary, Poland, Scotland, Serbia, Ukraine


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 6 — Reusable PPDA-Style Bar Chart Builder
# ══════════════════════════════════════════════════════════════

def build_benchmark_bar(
    df: pd.DataFrame,
    value_col: str,
    title: str,
    subtitle: str = '',
    x_title: str = '',
    fmt: str = '.1f',
    ascending: bool = True,
    top_n: int = 5,
    show_all: bool = False,
    highlight: str = HIGHLIGHT_TEAM,
) -> go.Figure:
    """
    PPDA-dashboard-style horizontal bar chart.
    Shows match count (GP) next to each team name so the reader
    always sees the sample-size context for a short tournament.

    Parameters
    ----------
    df          : DataFrame with columns ['team', 'matches', value_col]
    value_col   : column to plot on x-axis
    title       : chart title
    subtitle    : smaller text below title
    x_title     : x-axis label
    fmt         : number format for bar labels
    ascending   : sort direction (True = lowest on top)
    top_n       : how many teams to show (set 0 or show_all=True for all)
    show_all    : if True, show every team regardless of top_n
    highlight   : team name to highlight (default: Spain)
    """
    sorted_df = df.sort_values(value_col, ascending=ascending).copy()

    # Ensure highlight team is always included in top-N view
    if not show_all and top_n > 0:
        top = sorted_df.head(top_n)
        if highlight not in top['team'].values:
            highlight_row = sorted_df[sorted_df['team'] == highlight]
            top = pd.concat([top, highlight_row]).drop_duplicates(subset='team')
            top = top.sort_values(value_col, ascending=ascending)
        sorted_df = top

    # Reverse so that the "best" appears on top of the horizontal chart
    sorted_df = sorted_df.iloc[::-1]

    colors = [
        HIGHLIGHT_COLOR if t == highlight else NEUTRAL_COLOR
        for t in sorted_df['team']
    ]

    # Build y-axis labels with match count: "Spain (7 GP)"
    y_labels = [
        f'{t} ({int(m)} GP)' for t, m in zip(sorted_df['team'], sorted_df['matches'])
    ]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=sorted_df[value_col],
        y=y_labels,
        orientation='h',
        marker=dict(color=colors, line=dict(width=0)),
        text=sorted_df[value_col].apply(lambda v: f'{v:{fmt}}'),
        textposition='outside',
        textfont=dict(size=11, color='#c8d0d8'),
        hovertemplate=(
            '<b>%{y}</b><br>'
            + x_title + ': %{x:' + fmt + '}'
            + '<extra></extra>'
        ),
    ))

    full_title = title
    if subtitle:
        full_title += f'<br><sup style="color:rgba(255,255,255,0.55)">{subtitle}</sup>'

    fig.update_layout(
        template='plotly_dark',
        title=dict(text=full_title, font=dict(size=15, color='white')),
        xaxis=dict(
            title=x_title,
            gridcolor='rgba(255,255,255,0.06)',
            zeroline=False,
        ),
        yaxis=dict(title='', tickfont=dict(size=11)),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        height=max(340, len(sorted_df) * 38),
        margin=dict(l=170, r=80, t=65, b=40),
        showlegend=False,
        bargap=0.25,
    )
    return fig


print('✅ build_benchmark_bar() ready — PPDA-dashboard style')
print('   Y-axis labels now include match count (GP) for tournament context')

✅ build_benchmark_bar() ready — PPDA-dashboard style
   Y-axis labels now include match count (GP) for tournament context


---
## 1. GK Distribution Type — Short vs Long

### 1a. Short Distribution %
Teams ranked by the percentage of goal kicks played short (receiver in zones 1–6, own third).  
**Higher = more committed to short build-up from the GK.**

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 7 — KPI 1: Short Distribution %
# ══════════════════════════════════════════════════════════════

fig_short = build_benchmark_bar(
    kpi_df,
    value_col='short_pct',
    title='🏆 Short Distribution % — Top 5',
    subtitle=f'Euro 2024 · {HIGHLIGHT_TEAM} highlighted · Higher = more short build-up',
    x_title='Short %',
    ascending=False,   # highest on top
    top_n=5,
)
fig_short.show()

### 1b. Long Distribution %
Teams ranked by the percentage of goal kicks played long (receiver in zones 7–18, middle/attacking third).  
**Higher = more direct approach from the GK, bypassing the press.**

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 7b — KPI 1b: Long Distribution %
# ══════════════════════════════════════════════════════════════

fig_long = build_benchmark_bar(
    kpi_df,
    value_col='long_pct',
    title='🏆 Long Distribution % — Top 5',
    subtitle=f'Euro 2024 · {HIGHLIGHT_TEAM} highlighted · Higher = more direct / long build-up',
    x_title='Long %',
    ascending=False,   # highest on top
    top_n=5,
)
fig_long.show()

---
## 3. Positive Outcome % — Who Retains Possession Best?

Percentage of goal kicks where the team successfully retained possession (P1 + P2 + P3).  
**Higher = more effective at keeping the ball after GK distribution.**

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 8 — KPI 2: Positive Outcome %
# ══════════════════════════════════════════════════════════════

fig_pos = build_benchmark_bar(
    kpi_df,
    value_col='positive_pct',
    title='🏆 Positive Outcome % — Top 5',
    subtitle=f'Euro 2024 · {HIGHLIGHT_TEAM} highlighted · Higher = better possession retention',
    x_title='Positive %',
    ascending=False,
    top_n=5,
)
fig_pos.show()

---
## 4. Progression Rate (P2 + P3 %) — Who Reaches the Final Third?

Percentage of goal kicks where the team not only retained possession but progressed past x = 80 (final third) or created a shot.  
**Higher = more dangerous build-up from the back.**

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 9 — KPI 3: Progression Rate (P2 + P3 %)
# ══════════════════════════════════════════════════════════════

fig_prog = build_benchmark_bar(
    kpi_df,
    value_col='progression_pct',
    title='🏆 Progression Rate (P2 + P3 %) — Top 5',
    subtitle=f'Euro 2024 · {HIGHLIGHT_TEAM} highlighted · Higher = more dangerous build-up',
    x_title='Progression %',
    ascending=False,
    top_n=5,
)
fig_prog.show()

---
## 5. Danger Conceded (N2 + N3 %) — Who Is Most Vulnerable?

Percentage of goal kicks where the opponent entered the penalty area (N2) or created a shot (N3) after winning the ball.  
**Higher = more vulnerable from goal kicks.** This is the "risk" side of building from the back.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 10 — KPI 4: Danger Conceded (N2 + N3 %)
# ══════════════════════════════════════════════════════════════

fig_danger = build_benchmark_bar(
    kpi_df,
    value_col='danger_conceded_pct',
    title='⚠️ Danger Conceded (N2 + N3 %) — Top 5 Most Vulnerable',
    subtitle=f'Euro 2024 · {HIGHLIGHT_TEAM} highlighted · Higher = more vulnerable',
    x_title='Danger Conceded %',
    ascending=False,   # most vulnerable on top
    top_n=5,
)
fig_danger.show()

---
## 6. Full 6-Tier Outcome Breakdown — All Teams

Stacked horizontal bar showing the P1–P3 / N1–N3 split for every team, sorted by positive outcome rate. This gives the complete picture of how each nation's goal-kick build-up ended.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 11 — KPI 5: 6-Tier Outcome Stacked Bar (all teams)
# ══════════════════════════════════════════════════════════════

TIER_META = {
    'P1': ('#22c55e', 'Established Possession'),
    'P2': ('#16a34a', 'Reached Final Third'),
    'P3': ('#15803d', 'Created a Shot'),
    'N1': ('#f97316', 'Possession Lost'),
    'N2': ('#ef4444', 'Box Entry Conceded'),
    'N3': ('#dc2626', 'Shot Conceded'),
}

# Sort teams by positive_pct descending
sorted_kpi = kpi_df.sort_values('positive_pct', ascending=True).copy()

# Compute percentages for each tier
for code in ['P1', 'P2', 'P3', 'N1', 'N2', 'N3']:
    sorted_kpi[f'{code}_pct'] = (sorted_kpi[code] / sorted_kpi['total_gk'] * 100).round(1)

fig_tiers = go.Figure()

for code in ['P3', 'P2', 'P1', 'N1', 'N2', 'N3']:
    color, label = TIER_META[code]
    vals = sorted_kpi[f'{code}_pct'].tolist()

    fig_tiers.add_trace(go.Bar(
        y=sorted_kpi['team'],
        x=vals,
        orientation='h',
        name=f'{code} — {label}',
        marker_color=color,
        text=[f'{v:.0f}' if v >= 4 else '' for v in vals],
        textposition='inside',
        textfont=dict(size=9, color='white'),
        hovertemplate='<b>%{y}</b><br>' + f'{code} ({label})' + ': %{x:.1f}%<extra></extra>',
    ))

# Build y-axis labels with match count (GP) and 🇪🇸 for Spain
team_labels = sorted_kpi['team'].tolist()
highlighted_labels = []
for _, r in sorted_kpi.iterrows():
    t = r['team']
    gp = int(r['matches'])
    if t == HIGHLIGHT_TEAM:
        highlighted_labels.append(f'<b>🇪🇸 {t} ({gp} GP)</b>')
    else:
        highlighted_labels.append(f'{t} ({gp} GP)')

fig_tiers.update_layout(
    barmode='stack',
    template='plotly_dark',
    title=dict(
        text='🏆 6-Tier Outcome Breakdown — All Teams'
             '<br><sup style="color:rgba(255,255,255,0.55)">'
             'Euro 2024 · Sorted by Positive Outcome % · '
             'GP = Games Played · '
             'P1/P2/P3 (positive) · N1/N2/N3 (negative)</sup>',
        font=dict(size=15, color='white'),
    ),
    xaxis=dict(
        title='% of Goal Kicks',
        gridcolor='rgba(255,255,255,0.06)',
        zeroline=False,
        range=[0, 105],
    ),
    yaxis=dict(
        title='',
        tickfont=dict(size=10),
        ticktext=highlighted_labels,
        tickvals=team_labels,
    ),
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    height=max(550, len(sorted_kpi) * 28),
    margin=dict(l=170, r=30, t=70, b=40),
    legend=dict(
        orientation='h',
        yanchor='bottom', y=1.02,
        xanchor='center', x=0.5,
        font=dict(size=9),
    ),
    bargap=0.2,
)

fig_tiers.show()

---
## 7. Spain's Tournament Rank — Summary Card

Where does Spain sit in each KPI relative to the rest of the tournament?

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 12 — Spain's Rank Summary
# ══════════════════════════════════════════════════════════════

n_teams = len(kpi_df)
spain = kpi_df[kpi_df['team'] == HIGHLIGHT_TEAM].iloc[0]

rank_metrics = [
    ('Short Distribution %',   'short_pct',          False, spain['short_pct']),
    ('Long Distribution %',    'long_pct',           False, spain['long_pct']),
    ('Positive Outcome %',     'positive_pct',       False, spain['positive_pct']),
    ('Progression Rate %',     'progression_pct',    False, spain['progression_pct']),
    ('Danger Conceded %',      'danger_conceded_pct', True,  spain['danger_conceded_pct']),
]

print('═' * 65)
print(f'🇪🇸  SPAIN — TOURNAMENT RANKING SUMMARY')
print(f'   Euro 2024 · Benchmarked against {n_teams} teams')
print(f'   Spain: {int(spain["matches"])} GP ({spain["stage_reached"]}) — '
      f'{int(spain["total_gk"])} goal kicks analysed')
print('═' * 65)
print(f'\n⚠️  Sample-size note: teams played 3–7 matches.'
      f' GP shown in each chart.')

for label, col, lower_is_better, val in rank_metrics:
    if lower_is_better:
        rank = (kpi_df[col] < val).sum() + 1  # lower = better
        direction = '(lower is better)'
    else:
        rank = (kpi_df[col] > val).sum() + 1  # higher = better
        direction = '(higher is better)'

    medal = '🥇' if rank == 1 else '🥈' if rank == 2 else '🥉' if rank == 3 else '  '
    print(f'\n{medal} {label}')
    print(f'   Value: {val:.1f}%  |  Rank: {rank}/{n_teams} {direction}')

# ── Tournament averages ──
print(f'\n{"─" * 65}')
print(f'📊 TOURNAMENT AVERAGES (for context):')
for label, col, _, val in rank_metrics:
    avg = kpi_df[col].mean()
    median = kpi_df[col].median()
    diff = val - avg
    arrow = '▲' if diff > 0 else '▼' if diff < 0 else '='
    print(f'   {label:28s}: avg {avg:5.1f}% | median {median:5.1f}% | '
          f'Spain {val:5.1f}% ({arrow} {abs(diff):+.1f}pp)')

print(f'\n{"═" * 65}')
print(f'   Total goal kicks analysed: {len(df_dist):,}')
print(f'   Spain goal kicks: {int(spain["total_gk"])} across {int(spain["matches"])} matches ({spain["stage_reached"]})')

═════════════════════════════════════════════════════════════════
🇪🇸  SPAIN — TOURNAMENT RANKING SUMMARY
   Euro 2024 · Benchmarked against 24 teams
   Spain: 7 GP (Final) — 51 goal kicks analysed
═════════════════════════════════════════════════════════════════

⚠️  Sample-size note: teams played 3–7 matches. GP shown in each chart.

   Short Distribution %
   Value: 52.9%  |  Rank: 16/24 (higher is better)

   Long Distribution %
   Value: 47.1%  |  Rank: 9/24 (higher is better)

   Positive Outcome %
   Value: 47.1%  |  Rank: 11/24 (higher is better)

🥈 Progression Rate %
   Value: 19.6%  |  Rank: 2/24 (higher is better)

   Danger Conceded %
   Value: 9.8%  |  Rank: 7/24 (lower is better)

─────────────────────────────────────────────────────────────────
📊 TOURNAMENT AVERAGES (for context):
   Short Distribution %        : avg  57.6% | median  62.0% | Spain  52.9% (▼ +4.7pp)
   Long Distribution %         : avg  42.4% | median  38.0% | Spain  47.1% (▲ +4.7pp)
   Positive Outcome % 